# 第4回：教師あり学習基礎 〜 回帰分析の三段階発展と予測モデル 〜

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session04_supervised_learning.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

---

## 📋 本日の学習ロードマップ (計90分)

1. **【講義1】回帰分析の進化論：単回帰 から 重回帰 へ (15分)**
   - 1つの要因 vs 複数の複合要因。交絡因子の調整という概念。
2. **【実習1】単回帰・重回帰の実装と解釈 (15分)**
   - 偏回帰係数 ($β$) と P 値の読み方。どの項目が本当に重要か？
3. **【講義2】ロジスティック回帰：確率を予測する分類術 (15分)**
   - 直線を S 字カーブ（シグモイド）に曲げる。オッズ比との関係。
4. **【実習2】ロジスティック回帰による疾病予測 (15分)**
   - 確率 (Probability) と 判定 (Decision) の境界線。
5. **【講義3】モデル評価の三面鏡 (10分)**
   - R2 乗値、MSE、そして Accuracy。指標を使い分けるリテラシー。
6. **【実習3】機械学習の王道：決定木とランダムフォレスト (10分)**
   - 非線形な関係を捉える強力なアルゴリズムの体験。
7. **【総合演習】「臨床予測モデル (CPM)」の構築 (10分)**

---

## 1. セットアップ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm # 統計的な詳細解析に強いライブラリ
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style='whitegrid', context='talk')

print("回帰エンジン、出力全開。")

## 2. 【講義1】回帰の進化：単回帰 $\rightarrow$ 重回帰

### 2.1 単回帰 (Simple Linear Regression)
「A が増えれば B が増える」という一対一の関係。 $y = ax + b$ 。

### 2.2 重回帰 (Multiple Linear Regression) の必然性
医学や社会学では、1つの要因だけで結果が決まることは稀です。例えば「運動量」と「健康寿命」の関係を見たい時、「年齢」や「食事」といった他の要因（**交絡因子**）を無視すると、間違った結論になります。 
$$y = β_0 + β_1x_1 + β_2x_2 + ... + ε$$
他の変数を固定したまま、$x_1$ が 1 増えた時にどれだけ $y$ が変わるかを示すのが **偏回帰係数** です。

---

### 【実習1】重回帰による多変量解析の実装

In [ ]:
# 擬似データ：心血管リスクスコアの予測
np.random.seed(42)
n = 100
age = np.random.normal(55, 10, n)
bmi = np.random.normal(24, 4, n)
chol = np.random.normal(200, 30, n)
risk = 0.4 * age + 0.8 * bmi + 0.02 * chol + np.random.normal(0, 5, n)

X = pd.DataFrame({'Age': age, 'BMI': bmi, 'Chol': chol})
X_const = sm.add_constant(X) # 切片 (Intercept) の追加

# statsmodels を使うと、各変数の P 値（統計的有意性）が詳しく見えます
model_sm = sm.OLS(risk, X_const).fit()
print(model_sm.summary()) # 論文のような表が出ます

## 3. 【講義2】ロジスティック回帰：分類の数理

### なぜ「回帰」の名がつくのか？
内部の計算は重回帰と同じ $z = β_0 + β_1x_1 ...$ を行っていますが、その結果を **シグモイド関数** に通して $0 \sim 1$ に閉じ込めているためです。医学的には、この結果を「疾患の発生する確率」と解釈します。

---

### 【実習2】ロジスティック回帰による二値分類

In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X_cancer = pd.DataFrame(data.data, columns=data.feature_names).iloc[:, :5] # 最初の 5 項目のみ使用
y_cancer = data.target

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_cancer, y_cancer)

# 確率 (Proba) を表示してみる
probs = log_reg.predict_proba(X_cancer)[:5]
print("各データのクラス所属確率 (0:悪性, 1:良性):")
print(probs)

## 4. 【講義3】評価指標の正しい選び方

- **単・重回帰 (Regression)**: Mean Squared Error (MSE), R2-Score。
- **ロジスティック回帰 (Classification)**: Accuracy, Log-Loss, AUC。
「どれくらいズレているか」を重視するのか、「どれくらい正解カテゴリに当てたか」を重視するのかの使い分けを学びます。

---

## ✏️ 本日の最終研究ミッション (Regression Mastery)

**シナリオ**: あなたはある生活習慣病の予測モデルを構築しています。

### 課題 1: 交絡の罠から脱出せよ
「単回帰」で Age のみが Risk に与える影響を見た時と、「重回帰」で BMI も含めて Age の影響を見た時で、Age の係数 ($β$) はどのように変化しましたか？ その理由を「交絡」という言葉を使って説明してください。

### 課題 2: 確率の閾値計算
ロジスティック回帰において、AI が「悪性である確率 30% 以上」と判断した人をすべて陽性と判定するように、予測値を計算し直してください。

### 課題 3: 回帰から決定木へ
同じデータセットに対して `RandomForestRegressor` を適用し、重回帰（直線）では捉えられなかった精度の向上が見られるか、R2 スコアで比較してください。

---

In [ ]:
# ここに回答を記述してください



---
**Last updated**: 2026-02-15
**Instructor**: Nakaura-T (DS Department, Kumamoto Univ)